# K513 · Week 2, Session 2
## Data transformation — putting a table into the shape you need

On Tuesday you drew what was already in the table. Today you change the table: summarizing it,
reshaping it, and putting two of them together.

Four tools, and every one of them changes the shape of a table:

| | |
|---|---|
| `pivot_table()` and `groupby()` | summarize many rows into a few |
| `melt()` | values hiding in column headings become a real column |
| `concat()` | stack tables that are the same shape |
| `merge()` | bring columns from one table across to another |

Every chart you drew on Tuesday needed a column name for `x=`, `y=` and `hue=`. Today is how you get
a table that *has* those columns.

---

### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it. The same two things as Tuesday:

- It does not know your columns or what we covered in class. Whatever it writes, you own.
- **"Explain what this line does"** is the useful question. **"Write it for me"** is not.

The specific trap this session: an AI will happily write you a `merge()` that runs perfectly and
silently drops a third of your rows. Where a cell asks you *how many rows came back and why*, that
sentence is yours.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one. If
anything ever looks wrong: **Runtime → Restart session and run all**.

---
## 1 · Setup

Two real datasets from the course repo, and a few small tables typed out in full so you can see
every row of them. Run all four cells; only the last two show anything.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.precision', 3)

In [ ]:
BOSTON_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/BostonHousing.csv"
BIKE_URL   = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/bikeshare.csv"

boston_df = pd.read_csv(BOSTON_URL)
bike_df   = pd.read_csv(BIKE_URL)

print(boston_df.shape, bike_df.shape)

The small tables below are built with `pd.DataFrame`, which takes a dictionary: each key is a
column name and each value is the list of values down that column. You will not need to write one
of these from scratch, but it is worth reading once — it is the clearest possible statement of what
a DataFrame *is*.

In [ ]:
# The quarterly sales table from Week 1, Session 2 — the one you called untidy.
sales_df = pd.DataFrame({
    'store': ['Bloomington', 'Indianapolis'],
    'Q1':    [412, 388],
    'Q2':    [455, 401],
})

# The same idea with a full year and a third store.
sales_year_df = pd.DataFrame({
    'store': ['Bloomington', 'Indianapolis', 'Fort Wayne'],
    'Q1':    [412, 388, 296],
    'Q2':    [455, 401, 318],
    'Q3':    [331, 445, 340],
    'Q4':    [512, 605, 402],
})

# A pivot table of units sold: one row per quarter, one column per product.
quantity_df = pd.DataFrame({
    'Quarter':   ['Q1', 'Q2', 'Q3', 'Q4'],
    'Shirt':     [2500, 2700, 2600, 3000],
    'Sneakers':  [1500, 1600, 1700, 1800],
    'Shorts':    [4000, 4200, 4100, 4300],
    'Backpacks': [980, 1209, 2019, 1573],
})

sales_df

In [ ]:
# Customers, and the orders they have placed. Two tables, one shared column.
customer_df = pd.DataFrame({
    'CustomerID': [1111, 2222, 3333, 4444, 5555, 6666, 7777, 8888, 9999],
    'NAME':   ['Charles', 'Bertram', 'Barbara', 'Bala', 'Dennis',
               'Davis', 'Scott', 'John', 'Stephen'],
    'STREET': ['123 Main St.', '237 Ash Avenue', '111 Inwood St.', '119 Indiana St.',
               '333 Outwood Ave.', '444 Michigan St.', '777 State St.',
               '888 College Ave.', '768 Mall Rd.'],
    'ZIP':    [67226, 67226, 60606, 60616, 60606, 60616, 60616, 60606, 60606],
})

# The orders arrived as one file per year.
orders_2015 = pd.DataFrame({
    'OrderID':    [1020, 1021, 1022, 1023, 1024],
    'CustomerID': [1111, 1111, 2222, 3333, 3333],
    'RECEIVED':   ['12/4/2015', '12/5/2015', '12/6/2015', '12/7/2015', '12/9/2015'],
})

orders_2016 = pd.DataFrame({
    'OrderID':    [1825, 1826, 1827],
    'CustomerID': [4444, 5555, 6666],
    'RECEIVED':   ['7/20/2016', '7/21/2016', '7/22/2016'],
})

print(customer_df.shape, orders_2015.shape, orders_2016.shape)
customer_df

---
## 2 · Summarize — `pivot_table()` and `groupby()`

On Tuesday you drew `sns.barplot(data=bike_df, x='weather', y='num_shared')` and got three bars. Each
bar's height was the **mean of `num_shared` within that weather group** — seaborn grouped the rows and
averaged them, silently. Here you do it yourself, and keep the answer.

In [ ]:
bike_df.pivot_table(index='weather', values='num_shared')

That is Excel's pivot table. The mapping is exact:

| Excel | pandas |
|---|---|
| Rows | `index=` |
| Columns | `columns=` |
| Values | `values=` |
| Summarize value field by | `aggfunc=` |
| Grand Totals | `margins=True` |

One difference worth knowing before it embarrasses you: **Excel defaults to Sum, pandas defaults to
Mean.**

Add a second grouping column and you get a proper cross-tab.

In [ ]:
bike_df.pivot_table(index='weather', columns='busyday', values='num_shared')

Look at the `wet` / `1` cell: **NaN**.

Before you assume that is a hole in the data, check. There are 731 rows in this table and every one
of them has a `weather` value and a `busyday` value — nothing is missing. So the empty cell is not a
gap in the collection, it is **a combination that has never occurred**: in two years there has never
been a day that was both wet and busy. Rain is why days are not busy.

That distinction matters every time you build a cross-tab. A grid gives every combination a cell,
so an absence has somewhere to show up. A list of groups just leaves it out, and you never notice.

### A pivot table is untidy on purpose

Look at what `columns='busyday'` did: it took the *values* of `busyday` — 0 and 1 — and made them
column headings. That is exactly what rule 2 forbids.

It is still the right thing to do here, because a person is reading the table. **Tidy is a shape
that tools require, not a virtue.** When the reader is a human, a grid beats a list.

Which means there has to be a way back, for when the next reader is seaborn instead of a person.
That is `melt()`, and it is the next section.

### One more spelling to recognize

Search for "pandas average by category" and almost every answer will use `groupby()` instead. It is
the same idea, and for everything in this course the two do the same job — `pivot_table()` is the
one that also gives you `columns=` and `margins=True`.

In [ ]:
bike_df.groupby('weather')['num_shared'].mean()

`margins=True` adds Excel's Grand Totals. Here `aggfunc='count'` counts days rather than
averaging rides.

In [ ]:
bike_df.pivot_table(index='weather', columns='busyday', values='date',
                    aggfunc='count', margins=True)

### ✏️ Now You Try · 1

**Part (a)** — Average `num_shared` by `weather`. Then run it again with `aggfunc='median'`. Does the
ranking change?

In [ ]:
bike_df.pivot_table(index='____', values='____')

In [ ]:
bike_df.pivot_table(index='weather', values='num_shared', aggfunc='____')

**Part (b)** — Average `num_shared` by `weather` and `busyday`, with **weather down the side** and
**busyday across the top**.

In [ ]:
# your code here

**Part (c)** — Count the days instead of averaging them, with `margins=True` turned on.
How many wet days were there in total? How many busy days?

In [ ]:
# your code here

**Your answer to (a):** does the ranking change, and what does that tell you?

*(double-click to edit)*

**Your answer to (c):**

*(double-click to edit)*

**Part (d)** — One pivot table on `boston_df` showing the **mean `MEDV`**, the **median `RM`**
and the **max `CRIM`** for each value of `CHAS`.

This needs a *dictionary* in `aggfunc`: `{'column': 'statistic', ...}`.

In [ ]:
# your code here

**Your answer to (d):** what is the biggest difference between the two kinds of tract?

*(double-click to edit)*

---
## 3 · Reshape — `melt()`

Here is the table from the Week 1 *Is this tidy?* slide.

In [ ]:
sales_df

You said it breaks **rule 2**: each column should be one variable, and `Q1` and `Q2` are not
variables. They are *values* of a variable called quarter, hiding in the column headings. You also
said what the tidy version should look like — three columns (store, quarter, sales) and four rows.

`melt()` is that, in one line.

In [ ]:
sales_df.melt(id_vars='store', var_name='quarter', value_name='sales')

Four rows: 2 stores × 2 quarters. **Melting always multiplies** — `n` rows and `k` melted
columns give `n × k` rows. Nothing is aggregated and nothing is lost.

The four arguments:

- **`id_vars=`** — the columns that stay. Everything that identifies the row.
- **`value_vars=`** — the columns to stack. Omit it and pandas stacks everything else.
- **`var_name=`** — what to call the new column of old headings. Default `'variable'`.
- **`value_name=`** — what to call the new column of values. Default `'value'`.

Always set the last two. The defaults are legal column names that survive into every chart axis and
every merge you do afterwards, and `variable` tells nobody anything.

### Why bother

Here is the same idea with a full year and three stores.

In [ ]:
sales_year_df

Try to draw it. `sns.barplot` needs a column name for `x=`, a column name for `y=` and a
column name for `hue=`. In this table there is no `quarter` column and no `sales` column — there is
**nothing to pass**. That is not seaborn being fussy: a bar chart needs one row per bar, and this
table does not have one row per bar.

Melt it first.

In [ ]:
sales_long = sales_year_df.melt(id_vars='store', var_name='quarter', value_name='sales')
sales_long

In [ ]:
sns.barplot(data=sales_long, x='quarter', y='sales', hue='store')
plt.show()

**Bloomington is the only store that falls in Q3.** Its customers are students, and in Q3 the
students are not there.

Nothing was added by melting — the wide table held exactly the same twelve numbers. What changed is
that a shape a tool can consume is a shape you can *see*.

### ✏️ Now You Try · 2

**Part (a)** — `quantity_df` is a pivot table: one row per quarter, one column per product. Melt it
into three columns: `Quarter`, `Product`, `Quantity_Sold`.

In [ ]:
quantity_df

In [ ]:
quantity_long = quantity_df.melt(id_vars='____', var_name='____',
                                 value_name='____')
quantity_long

**Part (b)** — How many rows went in, and how many came out? Why that number?

In [ ]:
# your code here

**Your answer to (b):**

*(double-click to edit)*

**Part (c)** — Draw it: a `barplot` with `x='Quarter'`, `y='Quantity_Sold'` and
`hue='Product'`.

In [ ]:
# your code here

**Part (d)** — One product behaves differently from the other three. Which one — and could you
have spotted it in the wide table?

**Your answer to (d):**

*(double-click to edit)*

---
## 4 · Combine — `concat()` and `merge()`

Two different operations, and the question to ask first is: **am I adding more rows of the same
kind, or more facts about the rows I already have?**

- Rows go **down**, with `concat()`.
- Facts go **across**, with `merge()`.

### concat() — stacking rows

The orders arrived as one file per year. Same columns, different rows.

In [ ]:
order_df = pd.concat([orders_2015, orders_2016], ignore_index=True)
order_df

Three things:

1. The DataFrames go in a **list**, inside `[ ]`. They are variable names, so no quotes.
2. **`ignore_index=True`** renumbers the rows. Without it you get `0,1,2,3,4,0,1,2` — two rows
   numbered 0 — and every later lookup by row number is wrong. Run the next cell to see it.
3. If the data does not already say which file a row came from, add a column that says so
   **before** you stack. After the stack it is too late.

In [ ]:
pd.concat([orders_2015, orders_2016])   # no ignore_index — look at the row numbers

### merge() — bringing a column across

`customer_df` knows who they are. `order_df` knows what they bought. Neither knows both. The one
thing they share is `CustomerID`, and that is the entire basis of the join.

Before you merge, look at both tables properly.

In [ ]:
customer_df[['CustomerID', 'NAME']]

In [ ]:
order_df

**Before you run the next cell, write down a number.** `customer_df` has 9 rows and
`order_df` has 8. How many rows come back?

In [ ]:
merged = customer_df.merge(order_df, on='CustomerID')
print(merged.shape)
merged[['CustomerID', 'NAME', 'OrderID']]

**Eight rows — representing only six customers.**

Nine went in, eight came out, and it looks like almost nothing happened. In fact:

- **Three customers vanished.** Scott, John and Stephen have never ordered anything, and the
  default join keeps only rows that matched.
- **Two customers doubled.** Charles and Barbara have two orders each, so each appears twice.

No error. No warning. Nothing in the output mentions the three customers who just left your
analysis. If you had been asked for the average order value per customer, you would now be
computing it on the customers who ordered — which is not what you were asked.

`how=` controls what happens to the rows that did not match.

In [ ]:
for how in ['inner', 'left', 'right', 'outer']:
    result = customer_df.merge(order_df, on='CustomerID', how=how)
    print(f"how={how:8s} {len(result):3d} rows, {result['CustomerID'].nunique()} customers")

`'inner'` is the default and it is the only one that can silently lose rows. `'left'` — keep my
table intact, bring across whatever you can find — is what an analyst usually wants, and it is not
the default.

In [ ]:
left = customer_df.merge(order_df, on='CustomerID', how='left')
left[['CustomerID', 'NAME', 'OrderID', 'RECEIVED']]

Those `NaN`s are not a data error — **they are the answer**. Three customers have never
ordered, and that is the re-engagement list Marketing keeps asking for. One line finds them:

```python
left[left['OrderID'].isna()]
```

One mechanical thing to expect: `OrderID` was a whole number, and a column cannot hold both
integers and `NaN`, so pandas has converted it to a float and prints `1020.0`. Your data is fine.

### The check

**Two lines, every single time you merge.** There is no substitute for this.

In [ ]:
print('before:', customer_df.shape, order_df.shape)
merged = customer_df.merge(order_df, on='CustomerID', how='left')
print('after: ', merged.shape, '|', merged['CustomerID'].nunique(), 'distinct customers')

- **Fewer rows than the table you started with?** Unmatched rows were dropped. Decide whether
  you meant that.
- **More rows than either table?** The key is not unique on one side, so every match multiplied.
- **Exactly the number you expected?** Check the distinct keys too — you can lose three and gain
  two and land on the number you were hoping for.

A join that quietly drops a third of your rows never raises an error. The shape check is the only
thing standing between you and a wrong number in a meeting.

`indicator=True` turns the outer join into a diagnosis.

In [ ]:
check = customer_df.merge(order_df, on='CustomerID', how='outer', indicator=True)
check['_merge'].value_counts()

### ✏️ Now You Try · 3

**Part (a)** — Stack `orders_2015` and `orders_2016` with `concat()`. How many rows?

In [ ]:
# your code here

**Part (b)** — Inner merge `customer_df` and `order_df` on `CustomerID`. Print the shape
**before and after**, and count the distinct customers in the result.

In [ ]:
# your code here

**Your answer to (b):** how many customers went in, and how many came out?

*(double-click to edit)*

**Part (c)** — Do it again with `how='left'`. Which three customers appear now, and what is in
their `OrderID`?

In [ ]:
# your code here

**Your answer to (c):**

*(double-click to edit)*

**Part (d)** — Outer merge with `indicator=True`, then `value_counts()` the `_merge` column.
How many rows are `left_only`?

In [ ]:
# your code here

**Your answer to (d):**

*(double-click to edit)*

---
## Untidy, and the tool that fixes it

The whole session on one table. Start from **what is wrong with the table**, not from the function.

| What is wrong with the table | Rule | The tool |
|---|---|---|
| `Q1`, `Q2`, `Q3` as column headings | rule 2 | `melt()` |
| One kind of thing split across two tables | rule 1 | `merge()` |
| One table arriving as one file per year | — | `concat()` |
| Two values of one variable in a cell | rule 4 | `.str.split()` then `.explode()` |
| Two different variables in a cell | rule 4 | `.str.split(expand=True)` |
| Nothing — you want a summary a person will read | — | `pivot_table()` |

The last row is the odd one out on purpose: `pivot_table()` is the tool for when you *want* an
untidy table, because a person is going to read it.

---
## One value per cell — the two different fixes

Not on the live path this session, and not assessed. It is here because rule 4 gets broken in two
different ways and they need two different tools — and telling them apart is a question about
**meaning**, not about syntax.

**Same variable, listed more than once → becomes more ROWS.**

In [ ]:
books_df = pd.DataFrame({
    'Book':  ['A Tale of Two Cities', 'The Little Prince', "Harry Potter and the Philosopher's Stone"],
    'Genre': ['Historical fiction', 'Novella', 'Fantasy, Young adult, Adventure'],
})
books_df

In [ ]:
books_df['Genre'] = books_df['Genre'].str.split(', ')
books_exploded = books_df.explode('Genre')
books_exploded

Three books became five rows, with every other column duplicated. Now
`books_exploded['Genre'].value_counts()` is a meaningful question.

**Different variables in one cell → becomes more COLUMNS.**

This is Table B from the Week 1 *Is this tidy?* slide.

In [ ]:
contacts_df = pd.DataFrame({
    'customer': ['A. Rivera', 'L. Osei'],
    'contact':  ['rivera@x.com, 812-555-0134', 'osei@x.com, 812-555-0199'],
})
contacts_df[['email', 'phone']] = contacts_df['contact'].str.split(', ', expand=True)
contacts_df

An email address and a phone number are **not** two values of one variable, so they must not
become two rows — that would duplicate the customer and break rule 3 while fixing rule 4.

Ask whether the two things in the cell are the same kind of fact. Genres are. An email and a phone
number are not.

---
## That's it

You can now take a table that arrived in the wrong shape and put it in the right one. Specifically,
you:

- summarized a long table two ways, and know which shape to reach for
- saw that a wide table can show an absence that a long one hides
- fixed the exact table you called untidy in Week 1, in one line
- watched a chart become possible only after a reshape
- stacked two files, joined two tables, and **checked the shape either side**

### Before Tuesday

The **Week 2 assignment** is on Canvas, due **Sunday 11:59 pm**. Finish any Now You Try cells you
did not reach — the assignment assumes you did them.

Tuesday is **Week 3 Session 1: relationships between variables** — correlation done properly.
Pearson and Spearman, when each one lies, and what a p-value is actually telling you.